# Nassau Candy Distributor: Factory Reallocation & Shipping Optimization

**Project objective:** use order, product, region, and factory reference data to summarize performance and compare factory assignment scenarios.

**Data integrity note:** the supplied Ship Date values are 904–1,642 days after Order Date. They are not credible operational lead times. This analysis does not fit a model to that field; alternate-factory transit values are explicitly illustrative distance proxies.

Run top to bottom. Keep the CSV in the notebook working directory.

## 1. Load libraries and source data

In [ ]:
from pathlib import Path
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_FILE = Path('Nassau Candy Distributor.csv')
if not DATA_FILE.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        DATA_FILE = Path(next(iter(uploaded)))
    except ImportError:
        raise FileNotFoundError('Place Nassau Candy Distributor.csv beside this notebook.')

df = pd.read_csv(DATA_FILE, dtype={'Postal Code': str})
print('Shape:', df.shape)
display(df.head())

## 2. Data quality and cleaning
Trim text, parse the dates, standardize product hyphen spacing, and derive order month and gross margin. Rows are retained.

In [ ]:
print('Duplicate rows:', df.duplicated().sum())
print('Missing cells:', int(df.isna().sum().sum()))
for c in df.select_dtypes(include='object'):
    df[c] = df[c].str.strip()
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d-%m-%Y', errors='coerce')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%d-%m-%Y', errors='coerce')
df['Product Name'] = df['Product Name'].str.replace(r'\s*-\s*', ' - ', regex=True)
df['Order Month'] = df['Order Date'].dt.to_period('M').astype(str)
df['Profit Margin %'] = np.where(df['Sales'].ne(0), df['Gross Profit'] / df['Sales'] * 100, np.nan)
df['Lead Time (days)'] = (df['Ship Date'] - df['Order Date']).dt.days
print('Order dates:', df['Order Date'].min(), 'to', df['Order Date'].max())
print('Ship dates:', df['Ship Date'].min(), 'to', df['Ship Date'].max())
display(df[['Sales','Units','Gross Profit','Cost','Lead Time (days)']].describe().round(2))
print('Max |Sales - Cost - Gross Profit|:', (df['Sales'] - df['Cost'] - df['Gross Profit']).abs().max())

## 3. KPI summary and exploratory analysis

In [ ]:
def summarize(by):
    out = df.groupby(by, dropna=False).agg(Orders=('Order ID','nunique'), Units=('Units','sum'), Sales=('Sales','sum'), Profit=('Gross Profit','sum'))
    out['Margin %'] = out['Profit'] / out['Sales'] * 100
    return out.sort_values('Sales', ascending=False).round(2)

for field in ['Division', 'Region', 'Ship Mode', 'Country/Region']:
    print('\n', field)
    display(summarize(field))

display(summarize('Product Name'))

In [ ]:
monthly = df.groupby('Order Month')[['Sales','Gross Profit']].sum()
monthly.plot(figsize=(12,4), title='Monthly sales and gross profit')
plt.xticks(rotation=60); plt.tight_layout(); plt.show()

summarize('Product Name')['Sales'].sort_values().plot(kind='barh', figsize=(9,6), title='Sales by product')
plt.tight_layout(); plt.show()

## 4. Shipping date validation
This is a blocking data limitation for supervised lead-time prediction. Do not interpret the computed interval as real delivery performance.

In [ ]:
lead = df['Lead Time (days)'].dropna()
print('Lead time min / median / max:', lead.min(), lead.median(), lead.max())
print('Counts above 60 days:', int((lead > 60).sum()), 'of', len(lead))
df.groupby('Ship Mode')['Lead Time (days)'].agg(['count','median','mean','min','max']).round(1)

## 5. Factory mapping and distance-based scenario proxy
Mappings and coordinates below come from the project brief. Region centers are coarse illustrative proxies; the Haversine distance is straight-line miles, not road distance. Mode speeds are assumptions solely for relative scenarios.

In [ ]:
factories = {
    "Lot's O' Nuts": (32.881893, -111.768036), "Wicked Choccy's": (32.076176, -81.088371),
    'Sugar Shack': (48.11914, -96.18115), 'Secret Factory': (41.446333, -90.565487),
    'The Other Factory': (35.1175, -89.971107),
}
product_factory = {
    'Wonka Bar - Nutty Crunch Surprise': "Lot's O' Nuts", 'Wonka Bar - Fudge Mallows': "Lot's O' Nuts",
    'Wonka Bar - Scrumdiddlyumptious': "Lot's O' Nuts", 'Wonka Bar - Milk Chocolate': "Wicked Choccy's",
    'Wonka Bar - Triple Dazzle Caramel': "Wicked Choccy's", 'Laffy Taffy': 'Sugar Shack',
    'SweeTARTS': 'Sugar Shack', 'Nerds': 'Sugar Shack', 'Fun Dip': 'Sugar Shack',
    'Fizzy Lifting Drinks': 'Sugar Shack', 'Everlasting Gobstopper': 'Secret Factory',
    'Lickable Wallpaper': 'Secret Factory', 'Hair Toffee': 'The Other Factory',
    'Kazookles': 'The Other Factory', 'Wonka Gum': 'Secret Factory',
}
region_center = {'Pacific': (38,-120), 'Interior': (39,-100), 'Atlantic': (39,-76), 'Gulf': (30,-90)}
mode_speed_mph = {'Same Day':900, 'First Class':650, 'Second Class':450, 'Standard Class':300}
def miles(a,b):
    r=3958.8; lat1,lon1,lat2,lon2=np.radians([a[0],a[1],b[0],b[1]])
    h=np.sin((lat2-lat1)/2)**2+np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    return 2*r*np.arcsin(np.sqrt(h))

def compare_factories(product, region, ship_mode='Standard Class'):
    center=region_center[region]; baseline=product_factory[product]
    baseline_hours=miles(factories[baseline],center)/mode_speed_mph[ship_mode]*24
    result=[]
    for factory, coord in factories.items():
        distance=miles(coord,center); hours=distance/mode_speed_mph[ship_mode]*24
        result.append({'Factory':factory,'Distance proxy (mi)':distance,'Illustrative hours':hours,
                       'Current assignment':factory==baseline,
                       'Relative proxy improvement %':(baseline_hours-hours)/baseline_hours*100})
    return pd.DataFrame(result).sort_values('Illustrative hours')

compare_factories('Wonka Bar - Milk Chocolate','Pacific')

## 6. Candidate shortlist and export
Shortlist ranks alternate factory/region pairs by distance reduction and observed order volume. It is a research queue, not an approved relocation plan. Validate factory suitability, capacity, shipment-level routes, service levels, and actual freight costs before implementation.

In [ ]:
candidates=[]
for product in sorted(df['Product Name'].unique()):
    if product not in product_factory: continue
    current=product_factory[product]
    for region in sorted(df.loc[df['Product Name'].eq(product),'Region'].dropna().unique()):
        center=region_center.get(region)
        if center is None: continue
        current_miles=miles(factories[current],center)
        closest,closest_miles=min(((f,miles(c,center)) for f,c in factories.items()),key=lambda x:x[1])
        if closest != current:
            subset=df[(df['Product Name']==product)&(df['Region']==region)]
            candidates.append({'Product':product,'Region':region,'Current factory':current,'Closest factory':closest,
                'Orders':subset['Order ID'].nunique(),'Distance reduction proxy (mi)':current_miles-closest_miles,
                'Sales':subset['Sales'].sum(),'Gross profit':subset['Gross Profit'].sum()})
recommendations=pd.DataFrame(candidates).sort_values(['Distance reduction proxy (mi)','Orders'],ascending=False)
display(recommendations.head(20).round(1))
recommendations.to_csv('nassau_reassignment_candidates.csv',index=False)
df.to_csv('nassau_clean.csv',index=False)
print('Saved nassau_reassignment_candidates.csv and nassau_clean.csv')

## 7. Conclusion and next data request
This dataset supports sales, profit, product, and region summaries. It does not support credible delivery-time training or a true profit-optimized network design. Request corrected shipment timestamps, factory capacities and product eligibility, lane-level freight costs, actual customer coordinates, and service-level commitments. Once received, train and cross-validate regression candidates (for example linear regression, random forest, and gradient boosting) and report RMSE, MAE, and R² on a temporal holdout. Until then, use the app's distance-only scenarios as hypotheses for investigation.